# Comparación de modelos — touchdowns

## Qué responde este notebook

¿Qué modelo predice mejor `receiving_tds`? Es un conteo raro: la mayoría de los partidos el
receptor no anota. Por eso compiten tres tipos de modelo: los mismos de recepciones y yardas, que
usan error cuadrático; modelos de conteo (Poisson y Tweedie, lineales y de árboles); y un modelo de
dos etapas. Siguiendo la política de
[`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb), cada modelo elige sus
hiperparámetros y el ganador se elige por **deviance de Poisson en validación**; MAE no decide
porque predecir cero lo minimiza. Las variables se definen en el
[glosario](../../../docs/data/GLOSARIO_VARIABLES.md).

El notebook también explica, con un diagnóstico de calibración, por qué un XGBoost Tweedie con
hiperparámetros por defecto queda peor que el promedio histórico, y revisa si conviene entrenar solo con las semanas
activas. La historia de esta corrección está en [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)
y en [`docs/decisions/0004-metricas-de-seleccion.md`](../../../docs/decisions/0004-metricas-de-seleccion.md).

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from itertools import product
from xgboost import XGBRegressor

pd.set_option("display.width", 160)
sns.set_style("whitegrid")
AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_arbol = features.columnas_por_modelo("arbol")
features_ridge = features.columnas_por_modelo("ridge")
train_mask, val_mask, test_mask = experimentos.dividir_temporal(
    wr, range(2016, 2022), [2022, 2023], [2024, 2025]
)
y = wr["receiving_tds"]
print(f"receiving_tds: min={y.min()}, max={y.max()} -- {(y == 0).mean():.1%} de las filas en cero")

COLUMNAS = ["val_deviance_poisson", "val_d2_oos", "val_r2_oos", "val_sesgo", "val_pendiente_calibracion",
            "val_brier_anota", "val_auc_anota", "val_mae", "test_deviance_poisson", "test_d2_oos", "test_r2_oos"]

def tabla(resultados):
    filas = {n: {f"{p}_{k}": v for p in ("val", "test") for k, v in s[p].items()} for n, s in resultados.items()}
    return pd.DataFrame(filas).T[COLUMNAS].astype(float).sort_values("val_deviance_poisson")

receiving_tds: min=0, max=4 -- 81.8% de las filas en cero


## 1. Los modelos genéricos (error cuadrático)

Las mismas cinco familias de `4.2_modelos_recepciones_yardas.ipynb`. Dentro de cada familia, los
hiperparámetros se eligen por deviance de Poisson en validación.

In [2]:
resultados_genericos, _ = experimentos.entrenar_y_evaluar(
    wr, "receiving_tds", features_arbol, features_ridge, train_mask, val_mask, test_mask
)
tabla(resultados_genericos).round(4)

,val_deviance_poisson,val_d2_oos,val_r2_oos,val_sesgo,val_pendiente_calibracion,val_brier_anota,val_auc_anota,val_mae,test_deviance_poisson,test_d2_oos,test_r2_oos
XGBoost,0.6132,0.1272,0.0894,0.0209,0.9994,0.1292,0.7198,0.3038,0.6238,0.1375,0.0909
RandomForest,0.6154,0.1241,0.0870,0.0231,0.9856,0.1295,0.7167,0.3054,0.6309,0.1278,0.0820
HistGradientBoosting,0.6163,0.1227,0.0832,0.0238,0.8936,0.1294,0.7150,0.3037,0.6279,0.1319,0.0845
Ridge,0.6414,0.0870,0.0836,0.0209,0.9781,0.1299,0.7161,0.3046,0.7486,-0.0349,0.0909
Baseline,1.7183,-1.4458,-0.1164,0.0081,0.3069,0.1470,0.6379,0.2936,1.7830,-1.4651,-0.0704


## 2. Modelos de conteo

- **Poisson (lineal)**: predice el número esperado de eventos, nunca negativo, y supone que la
  varianza es igual al promedio.
- **Tweedie 1.5 (lineal)**: como Poisson pero admite más dispersión y muchos ceros; la potencia 1.5
  está entre Poisson (1) y Gamma (2).
- **XGBoost con objetivo `count:poisson` o `reg:tweedie`**: los mismos supuestos de conteo dentro de
  árboles, que además captan interacciones. Usan la misma grilla pequeña que el XGBoost genérico
  (profundidad 3, 4 o 6; 200 o 400 árboles; tasa 0.03 u 0.08), elegida por deviance en validación.
- **HistGradientBoosting con pérdida Poisson**: la misma grilla que su versión genérica.

Se agrega como referencia un XGBoost Tweedie con los **hiperparámetros por defecto** de la librería
(tasa 0.3, profundidad 6, 100 árboles), para diagnosticarlo en la sección 5.

In [3]:
X_arbol = wr[features_arbol]

def mejor_en_validacion(modelos):
    mejor = None
    for modelo in modelos:
        salida = experimentos.evaluar_modelo(modelo, X_arbol, y, train_mask, val_mask, test_mask)
        if mejor is None or salida["val"]["deviance_poisson"] < mejor["val"]["deviance_poisson"]:
            mejor = salida
    return mejor

def lineal(modelo):
    return Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler()), ("modelo", modelo)])

GRILLA_XGB = list(product((3, 4, 6), (200, 400), (0.03, 0.08)))
resultados_conteo = {
    "Poisson (lineal)": experimentos.evaluar_modelo(lineal(PoissonRegressor(max_iter=1000)), X_arbol, y, train_mask, val_mask, test_mask),
    "Tweedie 1.5 (lineal)": experimentos.evaluar_modelo(lineal(TweedieRegressor(power=1.5, max_iter=1000)), X_arbol, y, train_mask, val_mask, test_mask),
    "XGBoost count:poisson": mejor_en_validacion(
        XGBRegressor(objective="count:poisson", max_depth=d, n_estimators=n, learning_rate=lr, random_state=0, n_jobs=-1)
        for d, n, lr in GRILLA_XGB),
    "XGBoost reg:tweedie 1.5": mejor_en_validacion(
        XGBRegressor(objective="reg:tweedie", tweedie_variance_power=1.5, max_depth=d, n_estimators=n, learning_rate=lr, random_state=0, n_jobs=-1)
        for d, n, lr in GRILLA_XGB),
    "HGB loss=poisson": mejor_en_validacion(
        HistGradientBoostingRegressor(loss="poisson", max_iter=mi, max_depth=md, random_state=0)
        for mi, md in product((100, 200, 300), (None, 6, 10))),
    "XGBoost reg:tweedie 1.5, hiperparámetros por defecto": experimentos.evaluar_modelo(
        XGBRegressor(objective="reg:tweedie", tweedie_variance_power=1.5, random_state=0, n_jobs=-1),
        X_arbol, y, train_mask, val_mask, test_mask),
}
for nombre in ("XGBoost count:poisson", "XGBoost reg:tweedie 1.5"):
    p = resultados_conteo[nombre]["modelo"].get_params()
    print(f"{nombre}: max_depth={p['max_depth']}, n_estimators={p['n_estimators']}, learning_rate={p['learning_rate']}")
p = resultados_conteo["HGB loss=poisson"]["modelo"].get_params()
print(f"HGB loss=poisson: max_iter={p['max_iter']}, max_depth={p['max_depth']}")

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_glm/glm.py:268: OptimizeWarning: Unknown solver options: iprint
  opt_res = scipy.optimize.minimize(
/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_glm/glm.py:268: OptimizeWarning: Unknown solver options: iprint
  opt_res = scipy.optimize.minimize(


XGBoost count:poisson: max_depth=3, n_estimators=400, learning_rate=0.03
XGBoost reg:tweedie 1.5: max_depth=3, n_estimators=200, learning_rate=0.03
HGB loss=poisson: max_iter=100, max_depth=10


## 3. Modelo de dos etapas (*hurdle*)

Primera etapa: probabilidad de anotar al menos un touchdown. Segunda etapa, entrenada solo con los
partidos en que sí anotó: cuántos. La predicción es probabilidad × conteo esperado. No obliga a una
sola función a explicar a la vez «casi siempre cero» y «a veces dos o tres».

In [4]:
hurdle = experimentos.entrenar_hurdle(wr, "receiving_tds", features_arbol, train_mask, val_mask, test_mask)

## 4. Tabla unificada, ordenada por deviance de validación

`d2_oos` y `r2_oos` comparan contra la media de entrenamiento; `brier_anota` y `auc_anota` evalúan la
probabilidad de anotar al menos uno (P = 1 − e^(−predicción)); MAE se muestra solo como referencia.

In [5]:
todos = {**resultados_genericos, **resultados_conteo, "Hurdle": hurdle}
tabla_final = tabla(todos)
tabla_final.round(4)

,val_deviance_poisson,val_d2_oos,val_r2_oos,val_sesgo,val_pendiente_calibracion,val_brier_anota,val_auc_anota,val_mae,test_deviance_poisson,test_d2_oos,test_r2_oos
XGBoost count:poisson,0.6128,0.1278,0.0903,0.0196,1.0199,0.1292,0.7190,0.3032,0.6221,0.1399,0.0914
XGBoost,0.6132,0.1272,0.0894,0.0209,0.9994,0.1292,0.7198,0.3038,0.6238,0.1375,0.0909
XGBoost reg:tweedie 1.5,0.6143,0.1256,0.0897,0.0101,0.9976,0.1292,0.7170,0.2968,0.6252,0.1356,0.0895
HGB loss=poisson,0.6151,0.1244,0.0848,0.0197,0.8946,0.1293,0.7147,0.3006,0.6277,0.1322,0.0826
RandomForest,0.6154,0.1241,0.0870,0.0231,0.9856,0.1295,0.7167,0.3054,0.6309,0.1278,0.0820
HistGradientBoosting,0.6163,0.1227,0.0832,0.0238,0.8936,0.1294,0.7150,0.3037,0.6279,0.1319,0.0845
Hurdle,0.6206,0.1167,0.0788,0.0199,0.9047,0.1303,0.7090,0.3035,0.6334,0.1243,0.0765
Tweedie 1.5 (lineal),0.6215,0.1154,0.0786,0.0207,0.9388,0.1301,0.7142,0.3054,0.6314,0.1271,0.0846
Poisson (lineal),0.6277,0.1065,0.0772,0.0240,1.1420,0.1309,0.7135,0.3126,0.6382,0.1176,0.0821
Ridge,0.6414,0.0870,0.0836,0.0209,0.9781,0.1299,0.7161,0.3046,0.7486,-0.0349,0.0909


In [6]:
primero = tabla_final.index[0]
candidatos_xgb = [n for n in tabla_final.index if n.startswith("XGBoost") and "por defecto" not in n]
mejor_xgb = candidatos_xgb[0]
if primero == mejor_xgb:
    elegido = primero
    print(f"{primero} queda primero en deviance de validacion")
else:
    comparacion = experimentos.diferencia_bootstrap(
        wr[val_mask], y[val_mask], todos[primero]["pred"][val_mask], todos[mejor_xgb]["pred"][val_mask], "deviance_poisson")
    elegido = mejor_xgb if comparacion["empate"] else primero
    print(f"{primero} menos {mejor_xgb} en deviance de validacion = {comparacion['diferencia']:+.4f} "
          f"(IC 95%: {comparacion['ic_inferior']:+.4f} a {comparacion['ic_superior']:+.4f}) -> "
          f"{'empate' if comparacion['empate'] else 'diferencia real'}; se elige {elegido}")

cumple, detalle = experimentos.cumple_requisitos("receiving_tds", todos[elegido]["val"])
print(f"\nrequisitos de {elegido} en validacion: {'cumple' if cumple else 'NO cumple'}")
for chequeo, ok in detalle.items():
    print(f"   {'ok ' if ok else 'NO '} {chequeo}")

XGBoost count:poisson queda primero en deviance de validacion

requisitos de XGBoost count:poisson en validacion: cumple
   ok  R2 fuera de muestra > 0 (validacion)
   ok  D2 fuera de muestra > 0 (validacion)
   ok  sesgo adicional al de la media de entrenamiento <= 5% de la media (validacion)
   ok  pendiente de calibracion entre 0.9 y 1.1 (validacion)


**Lectura.** Los primeros quedan prácticamente empatados en deviance de validación: XGBoost con
objetivo Poisson (0.6128), el XGBoost genérico (0.6132), XGBoost Tweedie 1.5 (0.6143),
HistGradientBoosting con pérdida Poisson (0.6151) y RandomForest (0.6154). Los tres XGBoost tienen
D² de 0.13 y R² fuera de muestra de 0.09, por encima de la referencia de 0.06–0.07 que se calculó en
`4.1` bajo el supuesto Poisson. Es lo que se
anticipó ahí: como casi nunca hay más de un touchdown por partido, el azar real es menor que el de
una Poisson. En cualquier caso, el margen alcanzable es pequeño.

Gana XGBoost con objetivo Poisson (profundidad 3, 400 árboles, tasa 0.03), con pendiente de
calibración de 1.02, y cumple los requisitos. En validación, todos los modelos aprendidos, salvo la
configuración por defecto, superan al promedio histórico (D² > 0) y discriminan de forma parecida: su
AUC de «anota al menos uno» va de 0.709 a 0.720. Por la pendiente de calibración, que debe quedar
entre 0.9 y 1.1, no serían elegibles los dos HistGradientBoosting (0.895 y 0.894) ni la Poisson
lineal (1.142). Los extremos de la tabla tampoco: el baseline del promedio del jugador (D² −1.45)
predice exactamente cero a quien no ha anotado en la temporada, y la deviance castiga mucho predecir
cero cuando el evento ocurre. El XGBoost Tweedie por defecto (D² −0.19) tiene, en cambio, el MAE más
bajo de la tabla (0.240): es el problema de elegir por MAE que se describe en `4.1`. En prueba,
XGBoost con objetivo Poisson sigue primero (D² de 0.140 y R² de 0.091).

## 5. ¿Por qué fallan los hiperparámetros por defecto?

Gneiting y Resin (2023) descomponen el error cuadrático en tres partes: **incertidumbre** (la
variación propia del resultado, que no depende del modelo), **discriminación** (qué tanto separa el
modelo los partidos de más y menos touchdowns) y **descalibración** (qué tanto se alejan sus
predicciones de los promedios reales que les corresponden). Se cumple R² = (discriminación −
descalibración) / incertidumbre. La descalibración se estima recalibrando las predicciones con una
regresión isotónica y midiendo cuánto baja el error. Se calcula en validación, con R² contra la media
de validación, para el modelo elegido y para el XGBoost Tweedie por defecto.

In [7]:
def descomposicion(real, pred):
    real = np.asarray(real, dtype=float)
    pred = np.clip(np.asarray(pred, dtype=float), 0, None)
    recalibrada = IsotonicRegression(out_of_bounds="clip").fit(pred, real).predict(pred)
    incertidumbre = np.mean((real - real.mean()) ** 2)
    error, error_recalibrado = np.mean((real - pred) ** 2), np.mean((real - recalibrada) ** 2)
    descalibracion, discriminacion = error - error_recalibrado, incertidumbre - error_recalibrado
    return {"descalibracion (MCB)": descalibracion, "discriminacion (DSC)": discriminacion,
            "incertidumbre (UNC)": incertidumbre, "R2 = (DSC - MCB) / UNC": (discriminacion - descalibracion) / incertidumbre}

POR_DEFECTO = "XGBoost reg:tweedie 1.5, hiperparámetros por defecto"
pd.DataFrame({n: descomposicion(y[val_mask], todos[n]["pred"][val_mask]) for n in (elegido, POR_DEFECTO)}).T.round(5)

,descalibracion (MCB),discriminacion (DSC),incertidumbre (UNC),R2 = (DSC - MCB) / UNC
XGBoost count:poisson,0.00272,0.02077,0.20544,0.08785
"XGBoost reg:tweedie 1.5, hiperparámetros por defecto",0.01196,0.01366,0.20544,0.00824


In [8]:
def por_deciles(pred, real):
    t = pd.DataFrame({"predicho": np.asarray(pred, dtype=float), "real": np.asarray(real, dtype=float)})
    t["decil"] = pd.qcut(t["predicho"].rank(method="first"), 10, labels=range(1, 11))
    return t.groupby("decil", observed=True)[["predicho", "real"]].mean()

pd.concat({n: por_deciles(todos[n]["pred"][val_mask], y[val_mask]) for n in (elegido, POR_DEFECTO)}, axis=1).round(3)

XGBoost count:poisson        XGBoost reg:tweedie 1.5, hiperparámetros por defecto       
                   predicho   real                                             predicho   real
decil                                                                                         
1                     0.038  0.031                                              0.001    0.060
2                     0.076  0.084                                              0.004    0.095
3                     0.106  0.086                                              0.010    0.099
4                     0.135  0.105                                              0.021    0.138
5                     0.170  0.119                                              0.039    0.170
6                     0.213  0.171                                              0.064    0.140
7                     0.254  0.233                                              0.101    0.202
8                     0.297  0.272                                              0.153    0.311
9                     0.352  0.346                                              0.236    0.290
10                    0.484  0.483                                              0.467    0.425

**Lectura.** En validación, el modelo elegido tiene descalibración de 0.0027 y discriminación de
0.0208, lo que da R² de 0.088. El XGBoost Tweedie por defecto discrimina menos (0.0137) y su
descalibración es más de cuatro veces mayor (0.0120), casi tan grande como su discriminación: R² de
0.008 en validación (contra la media de validación) y de −0.022 en prueba (contra la media de
entrenamiento), con D² de −0.19 en validación.

Los deciles muestran por qué. En los cinco deciles más bajos predice entre 0.001 y 0.039 touchdowns
cuando la tasa real va de 0.060 a 0.170, y en el decil más alto predice 0.467 contra 0.425 reales.
Sus predicciones son demasiado extremas (pendiente 0.69): con tasa de aprendizaje alta y árboles
profundos, el modelo se ajusta de más a los partidos de entrenamiento. El elegido ordena los deciles igual
que la tasa real, que crece de decil en decil, pero queda por encima en ocho de los diez (por
ejemplo, 0.170 contra 0.119 en el decil 5; en el más alto, 0.484 contra 0.483). Es su sesgo de +0.020
en validación, que viene en buena parte de que la tasa de touchdowns bajó de 0.216 en entrenamiento a
0.193 en validación (`4.1`): por eso los requisitos miden el sesgo adicional al de la media de
entrenamiento.

## 6. ¿Conviene entrenar solo con las semanas activas?

Mismo criterio que en `4.2_modelos_recepciones_yardas.ipynb`: ambas versiones se evalúan sobre las
mismas semanas activas de **validación** (`targets > 0`), ahora con deviance de Poisson.

In [9]:
activos_mask = wr["targets"] > 0
val_activos_mask = val_mask & activos_mask
deviance_con_todas = {
    nombre: modeling.metricas_target("receiving_tds", y[val_activos_mask], salida["pred"][val_activos_mask])["deviance_poisson"]
    for nombre, salida in resultados_genericos.items()
}
_, resumen_solo_activos = experimentos.entrenar_y_evaluar(
    wr, "receiving_tds", features_arbol, features_ridge,
    train_mask & activos_mask, val_activos_mask, test_mask & activos_mask,
)
pd.DataFrame({
    "entrenado con todas (deviance en validacion activa)": deviance_con_todas,
    "entrenado solo con activas (deviance en validacion activa)": resumen_solo_activos["val_deviance_poisson"],
}).astype(float).round(4)

,entrenado con todas (deviance en validacion activa),entrenado solo con activas (deviance en validacion activa)
Baseline,1.9445,1.9451
HistGradientBoosting,0.6788,0.6821
RandomForest,0.6787,0.6799
Ridge,0.7101,0.6937
XGBoost,0.6761,0.6772


**Lectura.** Entrenar con todas las semanas gana en cuatro de los cinco modelos genéricos; la
excepción es Ridge (0.7101 contra 0.6937), el más débil de la comparación. Se entrena con todas las
semanas, igual que en recepciones y yardas.

## Cierre

En touchdowns, los mejores modelos de árboles quedan casi empatados, con pérdida de conteo o
genérica. Se elige XGBoost con objetivo Poisson, que cumple los requisitos y supera al promedio
histórico con D² de 0.13 en validación. El mal resultado de un XGBoost con hiperparámetros por
defecto viene de su descalibración, no del problema: eligiendo en validación con la métrica adecuada,
la R² fuera de muestra es positiva (0.09 en validación y en prueba) y cercana a lo que permite el
azar del touchdown. Sus hiperparámetros, y la potencia de Tweedie como alternativa, se afinan en
`4.4_ajuste_hiperparametros.ipynb`.

Anterior: [`4.2_modelos_recepciones_yardas.ipynb`](4.2_modelos_recepciones_yardas.ipynb) · Siguiente:
[`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb)